In [5]:
import os
import json
import pandas as pd
import numpy as np
from data_loader import load_all

# ==========================================================
# CONFIG
# ==========================================================
path_files = r"G:\CMPUT660Project\inputs\50prs\repo_month_complexity_cache"

data = load_all()
before_shas = set(data["commits_before"]["sha"])
after_shas = set(data["commits_after"]["sha"])
print(f"Total repos with 'before' commits: {len(set(data['commits_before']['repo']))}")
print(f"Total repos with 'after' commits: {len(set(data['commits_after']['repo']))}")
# We store the *AVERAGES* for each file here
file_averages = {
    "before": {"nloc": [], "ccn": [], "tokens": [], "params": [], "length": []},
    "after":  {"nloc": [], "ccn": [], "tokens": [], "params": [], "length": []}
}

metric_names = ["nloc", "ccn", "tokens", "params", "length"]

print("Calculating per-file averages...")
before_p = 0
after_p = 0

has_after = False
has_before = False
after = 0
before = 0
for fname in os.listdir(path_files):
    if not fname.endswith("_cache.json"):
        continue
        
    path = os.path.join(path_files, fname)
    try:
        with open(path, "r", encoding="utf-8") as f:
            cache_json = json.load(f)
        
        sha_to_metrics = cache_json.get("sha_to_metrics", {})
        
        # Temporary storage for functions in THIS specific file/repo
        current_file_funcs = {
            "before": {m: [] for m in metric_names},
            "after":  {m: [] for m in metric_names}
        }

        for commit, meta in sha_to_metrics.items():
            functions = meta.get('functions_info', [])
            if not functions or isinstance(functions, str):
                continue
            
            period = None
            if commit in before_shas:
                period = "before"
                before_p += 1
                if not has_before:
                    has_before = True
                    before += 1
            elif commit in after_shas:
                period = "after"
                after_p += 1
                if not has_after:
                    has_after = True
                    after += 1
            if period:
                for func in functions:
                    for m in metric_names:
                        val = func.get(m)
                        if val is not None:
                            current_file_funcs[period][m].append(val)

        # Now, calculate the MEAN for THIS file and add it to our master list
        for period in ["before", "after"]:
            for m in metric_names:
                vals = current_file_funcs[period][m]
                if vals:
                    # Append the mean of this file to the global list of file-means
                    file_averages[period][m].append(np.mean(vals))
        has_after = False  # reset for next file     
        has_before = False  # reset for next file         
    except Exception as e:
        print(f"Error processing {fname}: {e}")

# ==========================================================
# FINAL ANALYSIS (Average of Averages)
# ==========================================================
stats = []

for m in metric_names:
    row = {"Metric": m.upper()}
    
    # "Before" stats based on the distribution of file averages
    b_series = pd.Series(file_averages["before"][m]).dropna()
    row[("Before", "Mean")] = b_series.mean()
    row[("Before", "Median")] = b_series.median()
    
    # "After" stats based on the distribution of file averages
    a_series = pd.Series(file_averages["after"][m]).dropna()
    row[("After", "Mean")] = a_series.mean()
    row[("After", "Median")] = a_series.median()
    
    stats.append(row)

df_final = pd.DataFrame(stats)
df_final.columns = pd.MultiIndex.from_tuples(
    [("Metric", "")] + [c for c in df_final.columns if c != "Metric"]
)

print("\n" + "="*75)
print("CODE COMPLEXITY: MACRO-AVERAGE (Equal weight per repo)")
print("="*75)
print(df_final.round(2).to_string(index=False))
print("="*75)

print(f"\nTotal commits with metrics - Before: {before_p}, After: {after_p}")
print(f"Total files with 'after' metrics: {after}")
print(f"Total files with 'before' metrics: {before}")

Total repos with 'before' commits: 77
Total repos with 'after' commits: 79
Calculating per-file averages...

CODE COMPLEXITY: MACRO-AVERAGE (Equal weight per repo)
Metric Before         After       
         Mean Median   Mean Median
  NLOC  14.62  11.26  13.51  10.09
   CCN   2.87   2.12   2.86   2.16
TOKENS 106.50  73.55 102.08  76.49
PARAMS   1.13   1.02   1.06   0.82
LENGTH  18.67  14.25  16.93  12.12

Total commits with metrics - Before: 1210, After: 159
Total files with 'after' metrics: 52
Total files with 'before' metrics: 54
